# DBMF diversification overlay

Test whether DBMF improves diversification for a configurable static portfolio. The default core portfolio is `80% SPY + 100% LQD`; weights are notional exposures and do not need to sum to 100%.

Interpretation: portfolio returns are the weighted sum of ETF daily returns. There is no financing-cost model here, so exposures above 100% should be read as a clean strategy comparison rather than a fully specified implementation PnL.

In [ ]:
import pandas as pd

from alpha_lab.analytics.returns import cumulative_returns, drawdown
from alpha_lab.backtest.metrics import summary
from alpha_lab.data.loaders.yfinance import load_prices
from alpha_lab.portfolio.long_only import fixed_weight_returns
from alpha_lab.reporting.charts import drawdown_chart, equity_curve, heatmap_monthly

In [ ]:
START = "2019-05-01"
END = None

CORE_WEIGHTS = {"SPY": 0.8, "LQD": 1.0}
OVERLAY_TICKER = "DBMF"
OVERLAY_WEIGHTS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]

ROLLING_WINDOW_DAYS = 252

In [ ]:
tickers = list(CORE_WEIGHTS) + [OVERLAY_TICKER]
prices = load_prices(tickers, start=START, end=END).dropna(how="any")
returns = prices.pct_change().dropna(how="any")
prices.tail()

In [ ]:
portfolio_returns = {}
base_returns = fixed_weight_returns(returns, CORE_WEIGHTS)
portfolio_returns["core"] = base_returns

for overlay_weight in OVERLAY_WEIGHTS:
    if overlay_weight == 0:
        continue
    weights = CORE_WEIGHTS | {OVERLAY_TICKER: overlay_weight}
    name = f"core + {overlay_weight:.0%} {OVERLAY_TICKER}"
    portfolio_returns[name] = fixed_weight_returns(returns, weights)

portfolio_returns = pd.DataFrame(portfolio_returns).dropna()
portfolio_returns.tail()

In [ ]:
stats = pd.DataFrame({name: pd.Series(summary(series)) for name, series in portfolio_returns.items()})
stats.loc["CorrToCore"] = portfolio_returns.corrwith(portfolio_returns["core"])
stats.loc["FinalWealth"] = cumulative_returns(portfolio_returns).iloc[-1]
stats.loc["Worst1M"] = (1 + portfolio_returns).resample("ME").prod().sub(1).min()
stats

In [ ]:
delta_vs_core = stats.sub(stats["core"], axis=0).drop(columns="core")
delta_vs_core.loc[["CAGR", "AnnVol", "Sharpe", "MaxDD", "Calmar", "FinalWealth", "Worst1M"]]

In [ ]:
best_by_sharpe = stats.loc["Sharpe"].idxmax()
best_by_calmar = stats.loc["Calmar"].idxmax()
pd.Series({"best_by_sharpe": best_by_sharpe, "best_by_calmar": best_by_calmar})

In [ ]:
equity_curve(portfolio_returns)

In [ ]:
dd = drawdown(portfolio_returns)
dd.plot(title="Drawdowns")

In [ ]:
rolling_corr = portfolio_returns.drop(columns="core").rolling(ROLLING_WINDOW_DAYS).corr(portfolio_returns["core"])
rolling_corr.plot(title=f"Rolling {ROLLING_WINDOW_DAYS}D correlation to core")

In [ ]:
rolling_vol = portfolio_returns.rolling(ROLLING_WINDOW_DAYS).std() * (252**0.5)
rolling_vol.plot(title=f"Rolling {ROLLING_WINDOW_DAYS}D annualized volatility")

In [ ]:
drawdown_chart(portfolio_returns[best_by_sharpe])

In [ ]:
heatmap_monthly(portfolio_returns[best_by_sharpe])